# Wiki recommender

## 0. Introduction

## 1. Setup

In [1]:
import pandas as pd
import requests
import bs4
from time import sleep
import re
import numpy as np
import time

## 2. Data Collection (Crawling & Scraping)

In [2]:
class WikiScraper():
    
    def __init__(self, start_url_list, delay=3):
        self.headers = {'User-Agent': 'Recommender'}
        self.delay = delay # in seconds
        self.list_of_articles = []
        self.start_url_list = start_url_list
        
    def _fetch_article(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
        except requests.RequestException:
            return None
        if response.status_code != 200: return None

        parsed = bs4.BeautifulSoup(response.text, "html.parser")

        article = {"url": response.url}
        article["title"] = parsed.select_one("#firstHeading").get_text()
        article["content"] = " ".join(p.get_text() for p in parsed.select("#mw-content-text p"))
        links = self._extract_links(parsed)

        return article, links
        
    def _extract_links(self, parsed):
        links = parsed.find_all('a', attrs={'href': re.compile(r'^https://en\.wikipedia\.org/wiki/')})
        np.random.shuffle(links)
        return [link['href'] for link in links if ':' not in link['href'].split('/wiki/')[1]]
    
    def crawl(self, max_num_articles=50):
        stack = list(reversed(self.start_url_list))
        visited = set()
        
        while stack and len(self.list_of_articles)<max_num_articles:
            url = stack.pop()
            if url in visited:
                continue
            
            visited.add(url)
            
            result = self._fetch_article(url)
            time.sleep(self.delay)
            
            if result is None:
                continue
            
            article, links = result
            stack.extend(links)
            
            if len(article["content"].split()) < 200:
                continue

            self.list_of_articles.append(article)
            
        print(f"Downloaded {len(self.list_of_articles)} articles, visited {len(visited)} URLs")
            
    def get_articles(self):
        return self.list_of_articles              

In [3]:
scraper = WikiScraper(
    ["https://en.wikipedia.org/wiki/Machine_learning",
     "https://en.wikipedia.org/wiki/Jazz"],
    delay=0.5,
)
scraper.crawl(max_num_articles=40)

df = pd.DataFrame(scraper.get_articles())
df["n_words"] = df["content"].str.split().str.len()

print(df[["title", "n_words"]])
print(df["content"].iloc[0][:500])

Downloaded 40 articles, visited 57 URLs
                               title  n_words
0                   Machine learning     8855
1                               ISBN     3418
2                            Reading     2468
3                        Bibliomania      422
4               Book discussion club     1565
5                          Ira Levin      778
6                     Sliver (novel)      260
7                        Lady Godiva     2850
8                               Silk     5524
9                              Denim     1877
10                             Nîmes     2393
11                     Hugh Chisholm      408
12  Dictionary of National Biography     1675
13            London Review of Books      388
14                      The Observer     1579
15                     Freeview (UK)     3901
16                       Ono (Spain)      721
17                            Kosovo    12689
18                           Belarus     9482
19                             Wheat    